# Model 1: SentenceTransformer + Regression DNN

**Architecture:** `all-MiniLM-L6-v2` (frozen, 22M params) → 384-dim dense embedding → DNN head (1024-dim, 6 ResidualBlocks, ~13M trainable params)

**Target:** Beat BoW DNN baseline (MAE $46.49)

**Key technique:** Pre-compute embeddings once → train regression head only → fast training

## vast.ai Setup (chỉ chạy lần đầu khi thuê máy)

Sau khi `git clone` repo và `cd` vào đúng thư mục, mở terminal trên vast.ai và chạy:

```bash
pip install uv
uv sync
```

Sau đó khởi động lại Jupyter kernel rồi chạy các cell bên dưới.

In [ ]:
from pricer.items import Item
from pricer.sentence_transformer_model import SentTransRunner
from pricer.evaluator import evaluate, plot_training_history

## 1. Load Data

In [ ]:
train, val, test = Item.from_hub("SeanSunny/items_full")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

## 2. Setup Model

Pre-computing 800k embeddings with SentenceTransformer. This is the most time-consuming step (~10-15 min on GPU).

In [ ]:
runner = SentTransRunner(train, val[:1000])
runner.setup()

## 3. Train

Max 15 epochs with early stopping (patience=3). CosineAnnealingLR schedule.

In [ ]:
history = runner.train(epochs=15, patience=3)

## 4. Training History

In [ ]:
plot_training_history(history, title="SentenceTransformer + DNN")

## 5. Evaluate on 200 Test Samples

Using `evaluate()` from `pricer/evaluator.py` — same evaluation framework as all other models.

In [ ]:
evaluate(runner.inference, test)

## 6. Save Model Weights

In [ ]:
runner.save("sentence_transformer_model.pth")
print("Saved to sentence_transformer_model.pth")

# 1. Sanity check — inference trên trained runner
sample = test[0]
pred_original = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  ${sample.price:.2f}")
print(f"Predict: ${pred_original:.2f}")
print(f"Error:   ${abs(pred_original - sample.price):.2f}")
print()

# 2. Load roundtrip test — load lại từ .pth và so sánh kết quả
runner.load("sentence_transformer_model.pth")
pred_loaded = runner.inference(sample)
diff = abs(pred_original - pred_loaded)
assert diff < 0.01, f"Load mismatch! Before=${pred_original:.2f} After=${pred_loaded:.2f}"
print(f"Load roundtrip test PASSED. Diff: ${diff:.4f}")

In [ ]:
# Quick sanity check
sample = test[0]
pred = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  ${sample.price:.2f}")
print(f"Predict: ${pred:.2f}")
print(f"Error:   ${abs(pred - sample.price):.2f}")